In [1]:
import io
import requests
import pandas as pd

NB_URL = "https://data.norges-bank.no/api/data/EXR/B.{currencies}.NOK.SP"

def download_exchange_rates(start, end, currencies=("EUR", "USD")):
    """Daily NOK exchange rates from Norges Bank's open data API.

    start, end: dates as "YYYY-MM-DD"
    currencies: currency codes, e.g. ("EUR", "USD")
    Returns a dataframe with one row per business day and one column per
    currency (NOK per 1 unit of the currency).
    """
    url = NB_URL.format(currencies="+".join(currencies))
    params = {"format": "csv", "startPeriod": start, "endPeriod": end, "locale": "en"}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()

    raw = pd.read_csv(io.StringIO(resp.text), sep=None, engine="python")
    print("Columns in the response:", raw.columns.tolist())   # to check the format

    cols = {c.upper(): c for c in raw.columns}
    date_col, value_col, cur_col = cols["TIME_PERIOD"], cols["OBS_VALUE"], cols["BASE_CUR"]
    raw[date_col] = pd.to_datetime(raw[date_col])
    raw[value_col] = pd.to_numeric(raw[value_col], errors="coerce")
    table = raw.pivot(index=date_col, columns=cur_col, values=value_col).sort_index()
    table.index.name = "date"
    return table

# Test: EUR and USD during 2024
rates = download_exchange_rates("2024-01-01", "2024-12-31")
print(rates.shape)
print(rates.head())

Columns in the response: ['FREQ', 'Frequency', 'BASE_CUR', 'Base Currency', 'QUOTE_CUR', 'Quote Currency', 'TENOR', 'Tenor', 'DECIMALS', 'CALCULATED', 'UNIT_MULT', 'Unit Multiplier', 'COLLECTION', 'Collection Indicator', 'TIME_PERIOD', 'OBS_VALUE']
(251, 2)
BASE_CUR        EUR      USD
date                        
2024-01-02  11.2815  10.2971
2024-01-03  11.3200  10.3672
2024-01-04  11.2845  10.3027
2024-01-05  11.3090  10.3553
2024-01-08  11.3620  10.3800


In [2]:
import tomllib
import requests
import pandas as pd
from pymongo import MongoClient

# --- open-meteo: requested with timezone=UTC, so the hourly index is UTC ---
weather_no3 = pd.read_csv(
    "weather_cache/NO3_2024-01-01_2024-12-31_temperature_precipitation.csv",
    index_col=0, parse_dates=True)
weather = weather_no3.copy()
weather.index = weather.index.tz_localize("UTC")
local = weather.tz_convert("Europe/Oslo")

# Hours per local day on the two daylight saving changes in 2024
hours_per_day = local.groupby(local.index.date).size()
print("Spring change (31 Mar 2024):", hours_per_day[pd.Timestamp("2024-03-31").date()], "hours")
print("Autumn change (27 Oct 2024):", hours_per_day[pd.Timestamp("2024-10-27").date()], "hours")

# --- Norges Bank: one value per business day, no time of day ---
print("NB index:", rates.index.dtype, "| first dates:", list(rates.index[:3].date))

# --- Magasinstatistikk: weekly values with a date only ---
resp = requests.get("https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData",
                    timeout=30)
mag = pd.DataFrame(resp.json())
print("Magasinstatistikk dato_Id sample:", mag["dato_Id"].iloc[0], "| dtype:", mag["dato_Id"].dtype)

# --- ENTSO-E: timestamps stored in MongoDB (secrets are not printed) ---
with open("../.streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)
doc = MongoClient(secrets["mongo"]["uri"])["ind320"]["entsoe_flows"].find_one()
print("ENTSO-E ts sample:", doc["ts"], "| type:", type(doc["ts"]), "| tzinfo:", getattr(doc["ts"], "tzinfo", None))

Spring change (31 Mar 2024): 23 hours
Autumn change (27 Oct 2024): 25 hours
NB index: datetime64[us] | first dates: [datetime.date(2024, 1, 2), datetime.date(2024, 1, 3), datetime.date(2024, 1, 4)]
Magasinstatistikk dato_Id sample: 2020-11-15 | dtype: str
ENTSO-E ts sample: 2024-11-01 11:00:00 | type: <class 'datetime.datetime'> | tzinfo: None


In [5]:
import pandas as pd

coll = MongoClient(secrets["mongo"]["uri"])["ind320"]["entsoe_flows"]

# Which period do the data cover, and for which months?
first = coll.find_one(sort=[("ts", 1)])["ts"]
last = coll.find_one(sort=[("ts", -1)])["ts"]
print("Data covers:", first, "to", last)
print("Months with data:", sorted(
    {(d["year"], d["month"]) for d in coll.find({}, {"year": 1, "month": 1, "_id": 0})}))

# Rows per day for one connection during the first three days of the data
one = coll.find_one()
docs = list(coll.find(
    {"from_zone": one["from_zone"], "to_zone": one["to_zone"],
     "ts": {"$gte": first, "$lt": first + pd.Timedelta(days=3)}},
    {"ts": 1, "_id": 0}))
ts = pd.to_datetime(pd.Series([d["ts"] for d in docs]))
print("Rows:", len(ts), "| first timestamp:", ts.min() if len(ts) else None)
print(ts.dt.date.value_counts().sort_index() if len(ts) else "no rows")

Data covers: 2024-11-01 00:00:00 to 2026-10-30 23:00:00
Months with data: [(2024, 11), (2024, 12), (2025, 1), (2025, 2), (2025, 3), (2025, 4), (2025, 5), (2025, 6), (2025, 7), (2025, 8), (2025, 9), (2025, 10), (2025, 11), (2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4), (2026, 5), (2026, 6), (2026, 7), (2026, 8), (2026, 9), (2026, 10)]
Rows: 72 | first timestamp: 2024-11-01 00:00:00
2024-11-01    24
2024-11-02    24
2024-11-03    24
Name: count, dtype: int64


In [6]:
import pandas as pd

def rows_per_day(start, end):
    """Rows per day for one connection between two dates."""
    docs = list(coll.find(
        {"from_zone": one["from_zone"], "to_zone": one["to_zone"],
         "ts": {"$gte": pd.Timestamp(start), "$lt": pd.Timestamp(end)}},
        {"ts": 1, "_id": 0}))
    s = pd.to_datetime(pd.Series([d["ts"] for d in docs]))
    print(f"{start} to {end}: {len(s)} rows, duplicated timestamps: {int(s.duplicated().sum())}")
    print(s.dt.date.value_counts().sort_index().to_string(), "\n")

rows_per_day("2025-03-29", "2025-04-01")   # spring change: 30 Mar 2025
rows_per_day("2025-10-25", "2025-10-28")   # autumn change: 26 Oct 2025

2025-03-29 to 2025-04-01: 72 rows, duplicated timestamps: 0
2025-03-29    24
2025-03-30    24
2025-03-31    24 

2025-10-25 to 2025-10-28: 72 rows, duplicated timestamps: 0
2025-10-25    24
2025-10-26    24
2025-10-27    24 



## Time zone investigation

| Source | Time information | Time zone |
|--------|------------------|-----------|
| open-meteo | Hourly values, requested with `timezone=UTC` | UTC |
| ENTSO-E (MongoDB) | Hourly datetimes without tzinfo (MongoDB stores datetimes in UTC and pymongo returns them as naive datetimes) | UTC (checked: 24 rows per day on both daylight saving days in 2025, no duplicated timestamps) |
| Magasinstatistikk | One value per week (Sunday), date only | None (calendar date) |
| Norges Bank | One rate per business day, date only | None (calendar date) |

**Findings**
- The local day in Europe/Oslo has 23 hours on 31 March 2024 and 25 hours on 27 October 2024 because of daylight saving time. Hourly data in local time would therefore have a missing or duplicated hour.
- The ENTSO-E data in MongoDB are in UTC: every day has 24 hours, also on 30 March 2025 and 26 October 2025, and there are no duplicated timestamps.
- The daily and weekly sources have no time of day, so they can only be matched to hourly data by calendar date.

**Solution**
- Keep all hourly data in UTC and convert to local time only when showing it.
- Combine hourly and daily/weekly data by aggregating the hourly values to the UTC calendar date (or week) first. At weekly resolution a shift of one or two hours at the day boundary is negligible.